In [1]:
import json
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:

DATASET_DIR = Path(r"E:\STUDY\Cardiff University\CMT 501 Team Project\Project\archive\images")

TRAIN_DIR = DATASET_DIR.parent / "train"
VAL_DIR = DATASET_DIR.parent / "validation"
TEST_DIR = DATASET_DIR.parent / "test"


MODEL_DIR = Path(r"..\models")

MODEL_DIR.mkdir(exist_ok=True)

In [3]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMAGE_SIZE = (IMG_HEIGHT, IMG_WIDTH)

BATCH_SIZE = 32
SEED = 42

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 10500 files belonging to 30 classes.
Found 2250 files belonging to 30 classes.
Found 2250 files belonging to 30 classes.


In [5]:
CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

print(NUM_CLASSES)

30


In [6]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1),

    layers.RandomTranslation(0.1, 0.1)

])

In [7]:
normalization = layers.Rescaling(1./255)

In [8]:
cnn_model = models.Sequential([

    layers.Input(shape=(224,224,3)),

    data_augmentation,

    normalization,

    # Block 1
    layers.Conv2D(32,3,padding="same",activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Block 2
    layers.Conv2D(64,3,padding="same",activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Block 3
    layers.Conv2D(128,3,padding="same",activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Block 4
    layers.Conv2D(256,3,padding="same",activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Dropout(0.3),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256,activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES,activation="softmax")
])

In [9]:
cnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30)             │         7,710 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 463,838 (1.77 MB)

 Trainable params: 462,878 (1.77 MB)

 Non-trainable params: 960 (3.75 KB)

In [10]:
cnn_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [11]:
early_stop = callbacks.EarlyStopping(

    monitor="val_loss",

    patience=7,

    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.3,

    patience=3,

    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(

    filepath=str(MODEL_DIR/"cnn_best.keras"),

    monitor="val_accuracy",

    save_best_only=True
)

In [12]:
EPOCHS = 30

history = cnn_model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=[

        early_stop,

        reduce_lr,

        checkpoint
    ]
)

Epoch 1/30


c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


329/329 ━━━━━━━━━━━━━━━━━━━━ 640s 2s/step - accuracy: 0.1353 - loss: 3.1497 - val_accuracy: 0.0329 - val_loss: 3.4685 - learning_rate: 1.0000e-04
Epoch 2/30
329/329 ━━━━━━━━━━━━━━━━━━━━ 631s 2s/step - accuracy: 0.1940 - loss: 2.8812 - val_accuracy: 0.2564 - val_loss: 2.7272 - learning_rate: 1.0000e-04
Epoch 3/30
329/329 ━━━━━━━━━━━━━━━━━━━━ 622s 2s/step - accuracy: 0.2227 - loss: 2.7610 - val_accuracy: 0.2729 - val_loss: 2.6336 - learning_rate: 1.0000e-04
Epoch 4/30
329/329 ━━━━━━━━━━━━━━━━━━━━ 624s 2s/step - accuracy: 0.2525 - loss: 2.6683 - val_accuracy: 0.2738 - val_loss: 2.6130 - learning_rate: 1.0000e-04
Epoch 5/30
329/329 ━━━━━━━━━━━━━━━━━━━━ 625s 2s/step - accuracy: 0.2795 - loss: 2.5779 - val_accuracy: 0.2702 - val_loss: 2.6221 - learning_rate: 1.0000e-04
Epoch 6/30
329/329 ━━━━━━━━━━━━━━━━━━━━ 624s 2s/step - accuracy: 0.2955 - loss: 2.5061 - val_accuracy: 0.3369 - val_loss: 2.4344 - learning_rate: 1.0000e-04
Epoch 7/30
329/329 ━━━━━━━━━━━━━━━━━━━━ 625s 2s/step - accuracy: 0.30

In [13]:
cnn_model.save(MODEL_DIR/"cnn_final.keras")